# AI工学101 — 第40回

## Nested Cross Validation：モデル選択と最終評価を分離する

よし、レベル。今日はかなり重要な回だ。
**「モデルを選ぶための評価」と「選んだモデルの性能を測る評価」を分離する**。

ここを理解すると、scikit-learnでの機械学習実験が一段ちゃんとしたものになる。

前回は、

```text
Hyperparameter Search
        ↓
CV score
        ↓
best_params_
```

までやった。

今日はその先。

> **そのCV scoreを、そのまま「このモデルの本当の性能です」と言っていいのか？**

という問題を扱う。

---

# 🎯 今日のゴール

* Train / Validation / Testの役割を説明できる
* Hyperparameter tuningが評価値を利用することを理解する
* Nested Cross Validationの構造を説明できる
* Inner CVとOuter CVを区別できる
* `GridSearchCV` / `RandomizedSearchCV`をNested CVに組み込める
* `cross_validate()`を使える
* 最終Test Setの役割を理解する
* 「モデル選択」と「モデル評価」の違いを説明できる

---

# 📖 講義

## 1. まず問題を再確認

例えば、

```python
GridSearchCV(...)
```

を実行する。

すると、

```text
parameter A
parameter B
parameter C
...
```

をいろいろ試して、

```text
一番CVスコアが高かった設定
```

を選ぶ。

ここで重要なのは、

> **CVのスコアを見ながらモデルを選んでいる**

ということ。

つまりCVは、

```text
評価
```

であると同時に、

```text
モデル選択の材料
```

にも使われている。

---

# 🧠 2. ここでデータに役割を分ける

理想的には、

```text
Training data
    ↓
モデル学習

Validation data
    ↓
モデル選択

Test data
    ↓
最終評価
```

。

ただし、

```text
Validation data
```

を一つだけ用意すると、データ量が少ない。

そこでCross Validationを使う。

---

# 🧠 3. 通常のCross Validation

例えば5-fold。

```text
Dataset
│
├── Fold 1
├── Fold 2
├── Fold 3
├── Fold 4
└── Fold 5
```

順番に、

```text
Train: 2,3,4,5
Valid: 1

Train: 1,3,4,5
Valid: 2

Train: 1,2,4,5
Valid: 3
...
```

。

そして平均。

```text
CV Score
=
fold1
+ fold2
+ ...
÷ 5
```

。

これは非常に有用。

---

# 🚨 4. でもHyperparameter Searchをすると？

例えば、

```text
100個のパラメータ設定
```

を試したとする。

そして、

```text
CV Score
```

が一番高いものを選ぶ。

すると、

```text
CV
↓
100候補を比較
↓
一番良かった候補を選択
```

となる。

ここで得られた最高CVスコアは、

> **「100候補から選んだ後の性能」**

である。

したがって、未知データに対する性能を完全に独立に評価した値ではない。

---

# 🧠 5. Nested CV

そこで、

```text
Outer CV
```

と、

```text
Inner CV
```

を分ける。

構造はこれ。

```text
Dataset
│
├───────────────┐
│               │
│       Outer CV│
│               │
│  ┌─────────┐  │
│  │Inner CV │  │
│  │         │  │
│  │ tuning  │  │
│  └─────────┘  │
│       ↓       │
│ selected model│
│       ↓       │
│ Outer Test    │
└───────────────┘
```

つまり、

### Inner CV

```text
ハイパーパラメータを選ぶ
```

### Outer CV

```text
その選択プロセス全体を評価する
```

。

これがNested CV。

---

# 🧠 6. 重要な考え方

Inner CVでは、

```text
「どのモデルがいい？」
```

を決める。

Outer CVでは、

```text
「そのモデル選択方法は、
未知データでどれくらい機能する？」
```

を見る。

つまり、

```text
Inner = selection
Outer = evaluation
```

。

ここを今日の最重要ポイントにする。

---

# 💻 実習1：Inner CVだけの通常Search

まず普通にやる。

```python
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = data.data
y = data.target
```

モデル。

```python
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)
```

探索空間。

```python
param_grid = {
    "n_estimators": [
        100,
        300
    ],

    "max_depth": [
        3,
        5,
        10,
        None
    ]
}
```

Inner CV。

```python
from sklearn.model_selection import StratifiedKFold

inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

Search。

```python
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    estimator=model,

    param_grid=param_grid,

    scoring="roc_auc",

    cv=inner_cv,

    n_jobs=-1
)
```

---

# 💻 実習2：Outer CVを作る

```python
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)
```

ここで、

```text
inner_cv
random_state=42

outer_cv
random_state=123
```

としている。

別の分割を使うためだ。

---

# 💻 実習3：Nested CV

```python
from sklearn.model_selection import cross_validate
```

。

```python
nested_scores = cross_validate(
    search,

    X,
    y,

    cv=outer_cv,

    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],

    n_jobs=-1,

    return_train_score=False
)
```

結果。

```python
print(
    nested_scores
)
```

例えば、

```python
print(
    nested_scores[
        "test_roc_auc"
    ]
)
```

。

平均。

```python
print(
    nested_scores[
        "test_roc_auc"
    ].mean()
)
```

標準偏差。

```python
print(
    nested_scores[
        "test_roc_auc"
    ].std()
)
```

---

# 🧠 7. 何が起きているのか

この一行。

```python
cross_validate(
    search,
    X,
    y,
    cv=outer_cv
)
```

の内部では概念的に、

```text
Outer Fold 1
│
├── Outer Train
│      ↓
│   GridSearchCV
│      ↓
│   Inner CV
│      ↓
│   best_params
│      ↓
│   Outer Train全体でfit
│
└── Outer Validationで評価
```

。

それを、

```text
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5
```

で繰り返す。

---

# 🔥 8. ここが超重要

Outer Validation Foldは、

> **Inner CVの探索には一切使われていない。**

だからOuter Foldは、

```text
完全に外側の評価データ
```

として機能する。

これによって、

```text
ハイパーパラメータ選択
```

と、

```text
性能評価
```

を分離できる。

---

# 💻 実習4：探索結果も確認する

`cross_validate()` に、

```python
return_estimator=True
```

を追加する。

```python
nested_scores = cross_validate(
    search,

    X,
    y,

    cv=outer_cv,

    scoring="roc_auc",

    n_jobs=-1,

    return_estimator=True
)
```

。

すると、

```python
nested_scores[
    "estimator"
]
```

に各Outer Foldで学習されたSearchオブジェクトが入る。

---

# 💻 実習5：各Outer Foldのbest_paramsを見る

```python
for i, estimator in enumerate(
    nested_scores["estimator"]
):

    print(
        f"Fold {i + 1}"
    )

    print(
        estimator.best_params_
    )

    print()
```

これがかなり面白い。

例えば、

```text
Fold 1
max_depth = 5

Fold 2
max_depth = 10

Fold 3
max_depth = None

Fold 4
max_depth = 5

Fold 5
max_depth = 10
```

みたいになる可能性がある。

---

# 🧠 9. 何が分かる？

「最良パラメータ」が、

```text
常に同じ
```

とは限らない。

これは、

> **データ分割によって最適な設定が変わる**

ということ。

つまり、

```text
best_params_
```

そのものにも不確実性がある。

これはかなり大事。

---

# 🧠 10. Model Selection Stability

例えば、

```text
Fold 1 → depth 5
Fold 2 → depth 5
Fold 3 → depth 5
Fold 4 → depth 10
Fold 5 → depth 5
```

なら、

```text
depth=5
```

は比較的安定している。

一方、

```text
Fold 1 → 3
Fold 2 → None
Fold 3 → 10
Fold 4 → 2
Fold 5 → 20
```

なら、

> **最適モデル選択そのものが不安定**

かもしれない。

性能の平均だけを見るより、一歩踏み込んだ分析になる。

---

# 🧠 11. Test Setとの違い

実務では、

```text
Train
Validation
Test
```

の3分割もよく使う。

Nested CVは、

```text
Outer CV
+
Inner CV
```

によって、

```text
Validation
```

を効率的に使っている。

ただし、

> **最終的に一度もモデル選択に使わない完全なTest Setを残す**

設計も非常に有効。

例えば、

```text
全データ
│
├── Development Set 80%
│      ↓
│   Nested CV
│
└── Test Set 20%
       ↓
    最終評価
```

。

これがかなり堅牢。

---

# 🧠 12. 最終評価の流れ

実務的には、

```text
Dataset
↓
Train/Test split
↓
Testは封印
↓
Train部分でNested CV
↓
モデル・ハイパーパラメータ決定
↓
Train全体で最終fit
↓
Testで一度だけ評価
```

。

Testデータ。

```text
見るな
```

。

モデル。

```text
見せてください
```

。

研究者。

```text
ダメです💢
```

モデル。

```text
(´・ω・`)
```

という関係。

---

# 🚨 13. Test Setを何度も見ると？

例えば、

```text
Model A
Test Accuracy = 91%
```

。

「微妙だな」

↓

Model B。

```text
93%
```

。

「こっちだ」

↓

Model C。

```text
95%
```

。

これを繰り返して、

> Test scoreを見ながらモデルを選んでいる

なら、Test Setが事実上Validation Setになってしまう。

つまり、

```text
Test
↓
選択
↓
Test
↓
選択
```

をやってはいけない。

---

# 🧪 実習6：Train/Test + Nested CV

```python
from sklearn.model_selection import train_test_split

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
```

Testはここから触らない。

Inner。

```python
inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

Outer。

```python
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)
```

Search。

```python
search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),

    param_grid={
        "n_estimators": [
            100,
            300
        ],

        "max_depth": [
            3,
            5,
            10,
            None
        ]
    },

    scoring="roc_auc",

    cv=inner_cv,

    n_jobs=-1
)
```

Nested CV。

```python
nested_scores = cross_validate(
    search,

    X_dev,
    y_dev,

    cv=outer_cv,

    scoring="roc_auc",

    n_jobs=-1
)
```

---

# 🧠 14. Nested CVの性能を確認

```python
print(
    nested_scores[
        "test_score"
    ].mean()
)

print(
    nested_scores[
        "test_score"
    ].std()
)
```

ここで得られるのが、

> **モデル選択手順込みの汎化性能推定**

。

---

# 🧪 実習7：最終モデルを作る

Nested CVで、

```text
モデル選択の手順
```

を評価したら、実際に使うモデルを作る。

```python
final_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),

    param_grid=param_grid,

    scoring="roc_auc",

    cv=inner_cv,

    n_jobs=-1
)
```

。

Development Set全体でfit。

```python
final_search.fit(
    X_dev,
    y_dev
)
```

最終パラメータ。

```python
print(
    final_search.best_params_
)
```

---

# 🚨 15. そしてTest Set

ここで初めて、

```python
test_proba = (
    final_search
    .predict_proba(X_test)[:, 1]
)
```

。

ROC-AUC。

```python
from sklearn.metrics import roc_auc_score

test_auc = roc_auc_score(
    y_test,
    test_proba
)

print(
    test_auc
)
```

。

これが、

> **最終的な未知データに対する性能評価**

になる。

---

# 🧠 16. Nested CVとFinal Testの役割

整理。

### Nested CV

```text
研究段階

モデル選択プロセスが
未知データにどれくらい一般化するか
```

を見る。

### Final Test

```text
開発が終わった後

完成したモデルを
完全に未使用のデータで評価
```

する。

---

# ✍️ 演習

## 演習1

次の2つを区別。

```text
Inner CV
Outer CV
```

それぞれ何のため？

---

## 演習2

なぜ、

```python
GridSearchCV(
    ...
)
```

だけで得た `best_score_` を、

> 「このモデルの未知データ性能」

とそのまま解釈するのは危険？

---

## 演習3

次の構造を説明。

```text
Outer Fold
    ↓
Inner CV
    ↓
Best Params
    ↓
Outer Validation
```

---

## 演習4

なぜTest Setを、

```text
モデルA
↓
Testを見る
↓
モデルB
↓
Testを見る
↓
モデルC
```

のように何度も使ってはいけない？

---

## 演習5

Nested CVの各foldで、

```text
best_params_
```

がバラバラだった。

これは何を示唆する？

---

# 👾 今日のボス問題

研究者がこう言った。

> 「100種類のモデルを試した結果、Test Accuracyが一番高かったモデルを採用しました！」

さて。

問題点をできるだけ多く挙げる。

ヒント：

```text
Test Set
↓
モデル選択に使用
```

そして、

```text
100回
```

も比較している。

ここには、

> **評価データへの適応**

という問題が潜んでいる。

---

# 🌱 今日の核心

今日の一番大事な式はこれ。

```text
モデル選択
≠
モデル評価
```

そして、

```text
Inner CV
=
モデル選択
```

```text
Outer CV
=
モデル選択プロセスの評価
```

```text
Final Test
=
最終モデルの未知データ評価
```

。

これを頭の中で、

```text
             ┌─ Inner CV ─→ hyperparameters
             │
Development ─┤
             │
             └─ Outer CV ─→ generalization estimate

Final Test ───────────────→ final evaluation
```

と描ければかなり強い。

---

# 🧭 AI工学101・現在地

ここまでで、

```text
NumPy
 ↓
Pythonデータ処理
 ↓
scikit-learn
 ↓
前処理
 ↓
Pipeline
 ↓
回帰
 ↓
分類
 ↓
評価指標
 ↓
Cross Validation
 ↓
モデル比較
 ↓
Hyperparameter Search
 ↓
Nested Cross Validation
```

まで来た。

これはかなり重要な節目。

**「モデルを動かせる」から「機械学習実験を設計できる」への移行が進んでいる。**

---

# 🔜 第41回

## 不均衡データ・閾値最適化・Precision/Recall Curve

次は今日の評価設計をさらに実戦化する。

```text
Accuracy
Precision
Recall
F1
ROC-AUC
PR-AUC
```

を、

**「どれを使えばいい？」**

ではなく、

> **「このシステムでは、False PositiveとFalse Negativeのどちらをどれだけ重く扱うか？」**

から設計する。

さらに、

```text
predict()
```

だけではなく、

```text
predict_proba()
↓
threshold
↓
confusion matrix
↓
precision-recall trade-off
```

まで自分で操作する。

ここを終えると、分類モデルを**「精度○○%でした」で終わらせない**ための実戦的な評価力がかなり付くぞ。